In [ ]:
'''
Задание 1 (синтаксис)

Для каждого покупателя получите таблицу со столбцами:
 • preferred_payment - самый частый метод оплаты (при ничьей зафиксируйте правило, например lexicographically первый);
 • total_spent - сумма всех трат;
 • addons_spent - сумма на дополнительные услуги и аксессуары;
 • orders_count - число заказов.
 
 Отсортируйте по total_spent по убыванию и выведите топ-10.

Пример вывода (формат; числа зависят от датасета):

   customer_id  preferred_payment  total_spent  addons_spent  orders_count
 0         C042     Credit Card              15420.50        890.00             7
 1         C018     PayPal                     14810.00        450.00             5
 2         C103     Debit Card               13990.25       1200.00             6
 ...

Пограничные случаи:

• Покупатель с одним заказом - orders_count = 1, preferred_payment = единственный метод.
 • Два метода оплаты с одинаковой частотой - выберите один по зафиксированному правилу (укажите его в комментарии к коду).
 • Покупатель без доп. услуг/аксессуаров - addons_spent = 0.
 • Пропуски в сумме или методе оплаты - обработайте (отфильтруйте или заполните) и кратко опишите решение в комментарии.
'''

In [32]:
import pandas as pd

df = pd.read_csv('datasets/Electronic_sales_Sep2023-Sep2024.csv')

df = df.rename( columns = {
    "Customer ID":"customer_id",
    "Total Price":"total_price",
    "Add-on Total":"addon_total",
    "Payment Method":"payment_method"
    })

df = df.dropna(subset = ["customer_id", "total_price"])
df["addon_total"] = df["addon_total"].fillna(0)
df.head(10)

,customer_id,Age,Gender,Loyalty Member,Product Type,SKU,Rating,Order Status,payment_method,total_price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,addon_total
0,1000,53,Male,No,Smartphone,SKU1004,2,Cancelled,Credit Card,5538.33,791.19,7,2024-03-20,Standard,"Accessory,Accessory,Accessory",40.21
1,1000,53,Male,No,Tablet,SKU1002,3,Completed,Paypal,741.09,247.03,3,2024-04-20,Overnight,Impulse Item,26.09
2,1002,41,Male,No,Laptop,SKU1005,3,Completed,Credit Card,1855.84,463.96,4,2023-10-17,Express,NaN,0.00
3,1002,41,Male,Yes,Smartphone,SKU1004,2,Completed,Cash,3164.76,791.19,4,2024-08-09,Overnight,"Impulse Item,Impulse Item",60.16
4,1003,75,Male,Yes,Smartphone,SKU1001,5,Completed,Cash,41.50,20.75,2,2024-05-21,Express,Accessory,35.56
5,1004,41,Female,No,Smartphone,SKU1001,5,Completed,Credit Card,83.00,20.75,4,2024-05-26,Standard,"Impulse Item,Accessory",65.78
6,1005,25,Female,No,Smartwatch,SKU1003,3,Completed,Paypal,7603.47,844.83,9,2024-01-30,Overnight,NaN,0.00
7,1005,25,Female,No,Laptop,SKU1005,3,Completed,Debit Card,4175.64,463.96,9,2024-06-24,Overnight,"Extended Warranty,Extended Warranty",75.33
8,1006,24,Male,No,Smartphone,SKU1004,2,Cancelled,Debit Card,5538.33,791.19,7,2023-10-03,Standard,Impulse Item,43.05
9,1006,24,Male,Yes,Laptop,SKU1005,3,Completed,Cash,4175.64,463.96,9,2024-01-01,Express,NaN,0.00


In [48]:
pay = (
      df.dropna(subset=["payment_method"])
        .groupby(["customer_id","payment_method"]).size()
        .reset_index(name="cnt")
        .sort_values(["customer_id","cnt","payment_method"], ascending=[True, False, True])
        .drop_duplicates(subset="customer_id")
        .rename( columns = {
        "payment_method":"preferred_payment"
        })
        [["customer_id","preferred_payment"]]
    

)
pay.head(10)

,customer_id,preferred_payment
0,1000,Credit Card
2,1002,Cash
4,1003,Cash
5,1004,Credit Card
6,1005,Debit Card
8,1006,Cash
11,1007,Credit Card
12,1008,Cash
13,1011,Credit Card
14,1013,Paypal


In [51]:
df["spent"] = df["total_price"] + df["addon_total"]
agg = (
    df.groupby("customer_id")
      .agg(
          total_spent=("spent", "sum"),
          addons_spent=("addon_total", "sum"),
          orders_count=("spent", "size")
      )
      .reset_index())
agg.head(10)

,customer_id,total_spent,addons_spent,orders_count
0,1000,6345.72,66.30,2
1,1002,5080.76,60.16,2
2,1003,77.06,35.56,1
3,1004,148.78,65.78,1
4,1005,11854.44,75.33,2
5,1006,12317.70,133.43,3
6,1007,7176.19,55.48,1
7,1008,3445.17,65.85,1
8,1011,7982.07,70.17,1
9,1013,5596.14,57.81,1


In [52]:
res = agg.merge(pay, on="customer_id", how="left")
res["preferred_payment"] = res["preferred_payment"].fillna("Unknown")

res = (
    res[["customer_id", "preferred_payment", "total_spent", "addons_spent", "orders_count"]]
      .sort_values("total_spent", ascending=False)
      .head(10)
      .reset_index(drop=True)
)
print(res)

   customer_id preferred_payment  total_spent  addons_spent  orders_count
0        16357     Bank Transfer     35058.89        495.19             7
1        16863            PayPal     33522.09        486.17             5
2        13813       Credit Card     32098.80        268.64             5
3        11476            PayPal     31379.20        301.59             5
4        12276     Bank Transfer     31290.25        329.07             6
5        13635       Credit Card     30741.09        480.73             5
6        12749       Credit Card     30013.99        619.43             5
7        15399            PayPal     29390.27        305.39             3
8        19996     Bank Transfer     27728.90        432.12             6
9        12319     Bank Transfer     27578.70        226.38             3
